<a href="https://colab.research.google.com/github/PreethamHD/DP-MMFL/blob/main/notebooks/10_centralized_image_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import sys
from pathlib import Path
from google.colab import drive, userdata

# 1. Mount persistent storage (Google Drive)
if not os.path.exists("/content/drive"):
    drive.mount("/content/drive")

# 2. Configure Git identity and remote credentials
GITHUB_USERNAME = "PreethamHD"
REPO_NAME = "DP-MMFL"
REPO_DIR = Path(f"/content/{REPO_NAME}")
SRC_DIR = REPO_DIR / "src"

!git config --global user.name "PreethamHD"
!git config --global user.email "your_email@example.com"

try:
    token = userdata.get("GITHUB_TOKEN")
    repo_url = f"https://{token}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"
except Exception:
    repo_url = f"https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"

# 3. Clone repository if not present, or pull latest branch state
%cd /content
if not REPO_DIR.exists():
    !git clone {repo_url}
    %cd {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull origin main

# 4. Inject src directory to Python module search path
src_path_str = str(SRC_DIR.resolve())
if src_path_str not in sys.path:
    sys.path.insert(0, src_path_str)

# 5. Install required external dependencies
!pip install -q transformers pyarrow redivis

print("=" * 60)
print(f"Working Directory:  {os.getcwd()}")
print(f"Module Search Path: {src_path_str}")
print("Repository synchronized and dependencies installed successfully.")
print("=" * 60)

Mounted at /content/drive
/content
Cloning into 'DP-MMFL'...
remote: Enumerating objects: 148, done.
remote: Counting objects: 100% (148/148), done.
remote: Compressing objects: 100% (99/99), done.
remote: Total 148 (delta 61), reused 95 (delta 32), pack-reused 0 (from 0)
Receiving objects: 100% (148/148), 518.18 KiB | 14.39 MiB/s, done.
Resolving deltas: 100% (61/61), done.
/content/DP-MMFL
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.3/89.3 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 229.1/229.1 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 862.2/862.2 kB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.3/134.3 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 407.4/407.4 kB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 81.9 MB/s eta 0:00:00
Working Directory:  /content/DP-MMFL
Module Search Path: /content/DP-MMFL/src
Repository synchronized and dependencies install

In [ ]:
%%writefile /content/DP-MMFL/src/dp_mmfl/data/image_dataset.py
from io import BytesIO
from pathlib import Path
from typing import Optional, Union
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import Dataset
from dp_mmfl.data.labels import TARGET_COLUMNS


class CheXpertPlusImageDataset(Dataset):
    """
    Image-only PyTorch Dataset for CheXpert Plus frontal chest radiographs.
    Streams images on-demand from Redivis and provides 13 pathology targets with U-Ignore masks.
    """

    def __init__(
        self,
        manifest_path: Union[str, Path],
        table,
        split: str,
        image_transform=None,
    ):
        manifest = pd.read_parquet(manifest_path)

        manifest = manifest[
            (manifest["experiment_split"] == split)
            & (manifest["frontal_lateral"] == "Frontal")
        ].reset_index(drop=True)

        self.manifest = manifest
        self.table = table
        self.image_transform = image_transform

    def __len__(self) -> int:
        return len(self.manifest)

    def _load_image(self, path: str) -> Image.Image:
        png_path = str(path)

        if png_path.startswith("train/"):
            png_path = png_path[len("train/"):]

        if png_path.endswith(".jpg"):
            png_path = png_path[:-4] + ".png"

        file = self.table.file(png_path)
        data = file.read(as_text=False)

        image = Image.open(BytesIO(data)).convert("L")
        return image

    def __getitem__(self, index: int):
        row = self.manifest.iloc[index]

        image = self._load_image(row["path_to_image"])

        if self.image_transform is not None:
            image = self.image_transform(image)

        labels = torch.tensor(
            [
                0.0
                if pd.isna(row[f"target_{label}"])
                else float(row[f"target_{label}"])
                for label in TARGET_COLUMNS
            ],
            dtype=torch.float32,
        )

        label_mask = torch.tensor(
            [
                float(row[f"mask_{label}"])
                for label in TARGET_COLUMNS
            ],
            dtype=torch.float32,
        )

        return {
            "image": image,
            "labels": labels,
            "label_mask": label_mask,
            "sample_id": row["sample_id"],
            "patient_id": row["deid_patient_id"],
        }

Writing /content/DP-MMFL/src/dp_mmfl/data/image_dataset.py


In [ ]:
from google.colab import userdata

token = userdata.get("GITHUB_TOKEN")
repo_url = f"https://{token}@github.com/PreethamHD/DP-MMFL.git"

%cd /content/DP-MMFL
!git remote set-url origin {repo_url}
!git push origin main

/content/DP-MMFL
Enumerating objects: 10, done.
Counting objects: 100% (10/10), done.
Delta compression using up to 2 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (6/6), 1.30 KiB | 1.30 MiB/s, done.
Total 6 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To https://github.com/PreethamHD/DP-MMFL.git
   885c7d0..05b660a  main -> main


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path("/content/drive/MyDrive/DP-MMFL")
SRC_ROOT = Path("/content/DP-MMFL/src")

if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

MANIFEST_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "chexpert_plus_manifest.parquet"
)

print("Project root:   ", PROJECT_ROOT)
print("Manifest path:  ", MANIFEST_PATH)
print("Manifest exists:", MANIFEST_PATH.exists())
assert MANIFEST_PATH.exists(), "Manifest file not found!"

Project root:    /content/drive/MyDrive/DP-MMFL
Manifest path:   /content/drive/MyDrive/DP-MMFL/data/processed/chexpert_plus_manifest.parquet
Manifest exists: True


In [ ]:
import torch
from dp_mmfl.utils.seed import set_seed

SEED = 42
set_seed(SEED)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Seed:           ", SEED)
print("PyTorch version:", torch.__version__)
print("Device:         ", device)
if torch.cuda.is_available():
    print("GPU:            ", torch.cuda.get_device_name(0))

Seed:            42
PyTorch version: 2.11.0+cu130
Device:          cuda:0
GPU:             Tesla T4


In [ ]:
import pandas as pd

manifest = pd.read_parquet(MANIFEST_PATH)
print("Manifest shape:", manifest.shape)
print("\n--- Experiment Split Distribution (All Projections) ---")
print(manifest["experiment_split"].value_counts(dropna=False))

print("\n--- Experiment Split Distribution (Frontal Projections Only) ---")
print(
    manifest.loc[manifest["frontal_lateral"] == "Frontal", "experiment_split"]
    .value_counts(dropna=False)
)

Manifest shape: (223462, 58)

--- Experiment Split Distribution (All Projections) ---
experiment_split
train             178908
test               22563
val                21757
official_valid       234
Name: count, dtype: int64

--- Experiment Split Distribution (Frontal Projections Only) ---
experiment_split
train             152947
test               19349
val                18573
official_valid       202
Name: count, dtype: int64


In [ ]:
train_patients = set(
    manifest.loc[manifest["experiment_split"] == "train", "deid_patient_id"]
)
val_patients = set(
    manifest.loc[manifest["experiment_split"] == "val", "deid_patient_id"]
)
test_patients = set(
    manifest.loc[manifest["experiment_split"] == "test", "deid_patient_id"]
)

print("Train unique patients:", len(train_patients))
print("Val unique patients:  ", len(val_patients))
print("Test unique patients: ", len(test_patients))
print("Train ∩ Val:          ", len(train_patients & val_patients))
print("Train ∩ Test:         ", len(train_patients & test_patients))
print("Val ∩ Test:           ", len(val_patients & test_patients))

assert len(train_patients & val_patients) == 0, "Data leakage between Train and Val!"
assert len(train_patients & test_patients) == 0, "Data leakage between Train and Test!"
assert len(val_patients & test_patients) == 0, "Data leakage between Val and Test!"
print("\nPatient-level split isolation: PASS")

Train unique patients: 51620
Val unique patients:   6452
Test unique patients:  6453
Train ∩ Val:           0
Train ∩ Test:          0
Val ∩ Test:            0

Patient-level split isolation: PASS


In [ ]:
from torchvision import transforms

IMAGE_SIZE = 224
RESIZE_SIZE = 256

def get_train_transforms():
    return transforms.Compose([
        transforms.Resize(RESIZE_SIZE, antialias=True),
        transforms.RandomCrop(IMAGE_SIZE),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ])

def get_eval_transforms():
    return transforms.Compose([
        transforms.Resize(RESIZE_SIZE, antialias=True),
        transforms.CenterCrop(IMAGE_SIZE),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ])

print("Train and eval transforms configured (horizontal flip excluded).")

Train and eval transforms configured (horizontal flip excluded).


In [ ]:
import redivis
from dp_mmfl.data.image_dataset import CheXpertPlusImageDataset

table = redivis.table("aimi.chexpert_plus:5yyj:v1_0.png_train:s6cj")

train_transform = get_train_transforms()
eval_transform = get_eval_transforms()

train_dataset = CheXpertPlusImageDataset(
    manifest_path=MANIFEST_PATH,
    table=table,
    split="train",
    image_transform=train_transform,
)

val_dataset = CheXpertPlusImageDataset(
    manifest_path=MANIFEST_PATH,
    table=table,
    split="val",
    image_transform=eval_transform,
)

test_dataset = CheXpertPlusImageDataset(
    manifest_path=MANIFEST_PATH,
    table=table,
    split="test",
    image_transform=eval_transform,
)

print(f"Train frontal images: {len(train_dataset):,}")
print(f"Val frontal images:   {len(val_dataset):,}")
print(f"Test frontal images:  {len(test_dataset):,}")

# Integrity assertions on sample retrieval from each split
for name, ds in [("Train", train_dataset), ("Val", val_dataset), ("Test", test_dataset)]:
    sample = ds[0]
    assert sample["image"].shape == torch.Size([3, 224, 224]), f"{name}: image shape mismatch"
    assert sample["image"].dtype == torch.float32, f"{name}: image dtype mismatch"
    assert sample["labels"].shape == torch.Size([13]), f"{name}: labels shape mismatch"
    assert sample["label_mask"].shape == torch.Size([13]), f"{name}: label_mask shape mismatch"
    assert not torch.isnan(sample["labels"]).any(), f"{name}: NaN found in labels"
    assert not torch.isnan(sample["label_mask"]).any(), f"{name}: NaN found in label_mask"
    assert torch.all((sample["labels"] == 0.0) | (sample["labels"] == 1.0)), f"{name}: invalid label values"
    assert torch.all((sample["label_mask"] == 0.0) | (sample["label_mask"] == 1.0)), f"{name}: invalid mask values"
    print(f"[{name}] Sample check: PASS (Sample ID: {sample['sample_id']}, Patient: {sample['patient_id']})")

print("\nAll split datasets initialized and validated successfully.")

Train frontal images: 152,947
Val frontal images:   18,573
Test frontal images:  19,349
Please visit the URL below to authenticate with your Redivis account:
https://redivis.com/oauth/authorize?user_code=552f46a0535bf5441d22daac5df0b3b1
[Train] Sample check: PASS (Sample ID: 0, Patient: patient00003)
[Val] Sample check: PASS (Sample ID: 1, Patient: patient00007)
[Test] Sample check: PASS (Sample ID: 34, Patient: patient00055)

All split datasets initialized and validated successfully.


In [ ]:
%%writefile /content/DP-MMFL/src/dp_mmfl/models/image_model.py
import torch
import torch.nn as nn
from torchvision.models import (
    ResNet18_Weights,
    resnet18,
)


class ResNet18Classifier(nn.Module):
    """
    ResNet18 backbone adapted for 13-target multi-label pathology classification.
    Outputs raw unbounded logits (no internal sigmoid).
    """

    def __init__(
        self,
        num_classes: int = 13,
        pretrained: bool = True,
    ):
        super().__init__()

        weights = (
            ResNet18_Weights.DEFAULT
            if pretrained
            else None
        )

        self.backbone = resnet18(weights=weights)
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Linear(in_features, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.backbone(x)

Writing /content/DP-MMFL/src/dp_mmfl/models/image_model.py


In [ ]:
import os
os.makedirs("/content/DP-MMFL/src/dp_mmfl/evaluation", exist_ok=True)
Path("/content/DP-MMFL/src/dp_mmfl/evaluation/__init__.py").touch(exist_ok=True)

In [ ]:
%%writefile /content/DP-MMFL/src/dp_mmfl/evaluation/losses.py
import torch
import torch.nn.functional as F


def masked_bce_with_logits(
    logits: torch.Tensor,
    targets: torch.Tensor,
    mask: torch.Tensor,
) -> torch.Tensor:
    """
    Computes element-wise Binary Cross-Entropy with Logits, gates loss elements
    by binary mask (U-Ignore policy), and normalizes by the count of observed labels.
    """
    loss = F.binary_cross_entropy_with_logits(
        logits,
        targets,
        reduction="none",
    )

    masked_loss = loss * mask
    valid_count = mask.sum()

    if valid_count == 0:
        return masked_loss.sum() * 0.0

    return masked_loss.sum() / valid_count

Writing /content/DP-MMFL/src/dp_mmfl/evaluation/losses.py


In [ ]:
%cd /content/DP-MMFL
!git add src/dp_mmfl/models/image_model.py src/dp_mmfl/evaluation/
!git commit -m "added ResNet18Classifier and masked_bce_with_logits loss"
!git push origin main

/content/DP-MMFL
[main 6857072] added ResNet18Classifier and masked_bce_with_logits loss
 2 files changed, 59 insertions(+)
 create mode 100644 src/dp_mmfl/evaluation/losses.py
 create mode 100644 src/dp_mmfl/models/image_model.py
Enumerating objects: 13, done.
Counting objects: 100% (13/13), done.
Delta compression using up to 2 threads
Compressing objects: 100% (7/7), done.
Writing objects: 100% (8/8), 1.25 KiB | 1.25 MiB/s, done.
Total 8 (delta 4), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (4/4), completed with 4 local objects.
To https://github.com/PreethamHD/DP-MMFL.git
   05b660a..6857072  main -> main


In [ ]:
import torch
from dp_mmfl.models.image_model import ResNet18Classifier

model = ResNet18Classifier(
    num_classes=13,
    pretrained=True,
)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print("Device:            ", device)
print("Classifier Head:   ", model.backbone.fc)
print("Input Features:    ", model.backbone.fc.in_features)
print("Output Classes:    ", model.backbone.fc.out_features)

assert model.backbone.fc.in_features == 512, "Backbone feature dimension mismatch!"
assert model.backbone.fc.out_features == 13, "Output class dimension mismatch!"

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 189MB/s]


Device:             cuda:0
Classifier Head:    Linear(in_features=512, out_features=13, bias=True)
Input Features:     512
Output Classes:     13


In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0,
)

batch = next(iter(train_loader))
images = batch["image"].to(device)
labels = batch["labels"].to(device)
label_mask = batch["label_mask"].to(device)

print("Batch Images Input Shape: ", images.shape)

model.eval()
with torch.no_grad():
    logits = model(images)

print("Output Logits Shape:      ", logits.shape)
print("Output Logits Dtype:      ", logits.dtype)

assert logits.shape == torch.Size([4, 13]), f"Expected [4, 13], got {logits.shape}"
assert logits.dtype == torch.float32, f"Expected torch.float32, got {logits.dtype}"
assert not torch.isnan(logits).any(), "NaN values found in forward-pass logits!"
print("Forward-pass sanity check: PASS")

Batch Images Input Shape:  torch.Size([4, 3, 224, 224])
Output Logits Shape:       torch.Size([4, 13])
Output Logits Dtype:       torch.float32
Forward-pass sanity check: PASS


In [ ]:
from dp_mmfl.evaluation.losses import masked_bce_with_logits

# Compute loss using forward-pass outputs and the retrieved batch
loss_val = masked_bce_with_logits(logits, labels, label_mask)

print("Masked BCE Loss: ", loss_val.item())
print("Loss Tensor Dim: ", loss_val.shape)

assert loss_val.ndim == 0, "Loss is not a scalar!"
assert not torch.isnan(loss_val), "Computed loss is NaN!"
assert loss_val.item() > 0.0, "Computed loss must be positive!"
print("Masked BCE loss computation: PASS")

Masked BCE Loss:  0.5657671093940735
Loss Tensor Dim:  torch.Size([])
Masked BCE loss computation: PASS


Checking redvis->dataloader->gpu pipeline

In [ ]:
from torch.utils.data import DataLoader

BATCH_SIZE = 32
NUM_WORKERS = 0

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

print("Batch size:", BATCH_SIZE)
print("Workers:", NUM_WORKERS)
print("Batches per epoch:", len(train_loader))

Batch size: 32
Workers: 0
Batches per epoch: 4780


In [ ]:
import time
import torch

NUM_BENCHMARK_BATCHES = 20

model.eval()

batch_times = []
transfer_times = []

loader_iter = iter(train_loader)

for batch_idx in range(NUM_BENCHMARK_BATCHES):

    start = time.perf_counter()

    batch = next(loader_iter)

    data_ready = time.perf_counter()

    images = batch["image"].to(
        device,
        non_blocking=True,
    )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    transfer_done = time.perf_counter()

    batch_times.append(
        data_ready - start
    )

    transfer_times.append(
        transfer_done - data_ready
    )

    print(
        f"Batch {batch_idx + 1:02d}/{NUM_BENCHMARK_BATCHES}: "
        f"load={batch_times[-1]:.3f}s, "
        f"GPU transfer={transfer_times[-1]:.3f}s"
    )

Batch 01/20: load=24.189s, GPU transfer=0.002s
Batch 02/20: load=22.754s, GPU transfer=0.002s
Batch 03/20: load=21.235s, GPU transfer=0.002s
Batch 04/20: load=21.777s, GPU transfer=0.002s
Batch 05/20: load=23.048s, GPU transfer=0.002s
Batch 06/20: load=22.022s, GPU transfer=0.002s
Batch 07/20: load=21.086s, GPU transfer=0.002s
Batch 08/20: load=36.806s, GPU transfer=0.002s
Batch 09/20: load=23.090s, GPU transfer=0.002s
Batch 10/20: load=20.355s, GPU transfer=0.002s
Batch 11/20: load=22.474s, GPU transfer=0.002s
Batch 12/20: load=22.495s, GPU transfer=0.002s
Batch 13/20: load=21.798s, GPU transfer=0.002s
Batch 14/20: load=22.206s, GPU transfer=0.002s
Batch 15/20: load=22.777s, GPU transfer=0.002s
Batch 16/20: load=21.099s, GPU transfer=0.002s
Batch 17/20: load=20.862s, GPU transfer=0.002s
Batch 18/20: load=22.067s, GPU transfer=0.002s
Batch 19/20: load=22.026s, GPU transfer=0.002s
Batch 20/20: load=31.278s, GPU transfer=0.002s


In [ ]:
import numpy as np

avg_load_time = np.mean(batch_times)
avg_transfer_time = np.mean(transfer_times)

samples_per_second = (
    BATCH_SIZE / avg_load_time
)

print("\n--- Throughput Summary ---")
print(f"Average batch load time: {avg_load_time:.3f}s")
print(f"Average GPU transfer time: {avg_transfer_time:.3f}s")
print(f"Approx. samples/sec: {samples_per_second:.2f}")
print(
    f"Approx. batches/sec: "
    f"{1 / avg_load_time:.3f}"
)


--- Throughput Summary ---
Average batch load time: 23.272s
Average GPU transfer time: 0.002s
Approx. samples/sec: 1.38
Approx. batches/sec: 0.043


In [ ]:
num_train_samples = len(train_dataset)

estimated_epoch_seconds = (
    len(train_loader) * avg_load_time
)

estimated_epoch_minutes = (
    estimated_epoch_seconds / 60
)

estimated_epoch_hours = (
    estimated_epoch_minutes / 60
)

print("\n--- Estimated Epoch Time ---")
print("Training samples:", num_train_samples)
print("Training batches:", len(train_loader))
print(
    f"Estimated epoch time: "
    f"{estimated_epoch_minutes:.2f} minutes"
)
print(
    f"Estimated epoch time: "
    f"{estimated_epoch_hours:.2f} hours"
)


--- Estimated Epoch Time ---
Training samples: 152947
Training batches: 4780
Estimated epoch time: 1854.02 minutes
Estimated epoch time: 30.90 hours


In [ ]:
images = batch["image"].to(
    device,
    non_blocking=True,
)

# Warm-up
with torch.no_grad():
    for _ in range(5):
        _ = model(images)

if torch.cuda.is_available():
    torch.cuda.synchronize()

inference_times = []

for _ in range(20):

    start = time.perf_counter()

    with torch.no_grad():
        logits = model(images)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    end = time.perf_counter()

    inference_times.append(end - start)

avg_inference_time = np.mean(inference_times)

print(
    f"Average ResNet18 inference time: "
    f"{avg_inference_time:.4f}s"
)

print(
    f"Approx. inference throughput: "
    f"{BATCH_SIZE / avg_inference_time:.2f} samples/sec"
)

Average ResNet18 inference time: 0.0282s
Approx. inference throughput: 1136.51 samples/sec


In [ ]:
import shutil

print("=== Storage Information ===")

for path, label in [
    ("/content", "Colab local storage"),
    ("/content/drive", "Google Drive"),
]:
    total, used, free = shutil.disk_usage(path)

    print(f"\n{label}")
    print(f"Total: {total / 1024**3:.2f} GB")
    print(f"Used:  {used / 1024**3:.2f} GB")
    print(f"Free:  {free / 1024**3:.2f} GB")

=== Storage Information ===

Colab local storage
Total: 112.64 GB
Used:  42.61 GB
Free:  70.01 GB

Google Drive
Total: 112.64 GB
Used:  46.13 GB
Free:  66.51 GB


In [ ]:
from pathlib import Path

CACHE_ROOT = Path(
    "/content/dp_mmfl_image_cache"
)

print("Cache exists:", CACHE_ROOT.exists())

if CACHE_ROOT.exists():
    print("Existing cache size will be checked next.")

Cache exists: False


In [ ]:
from pathlib import Path

CACHE_ROOT = Path("/content/dp_mmfl_image_cache")
TRAIN_CACHE = CACHE_ROOT / "train"

TRAIN_CACHE.mkdir(parents=True, exist_ok=True)

print("Cache:", CACHE_ROOT)
print("Train cache:", TRAIN_CACHE)
print("Exists:", TRAIN_CACHE.exists())

Cache: /content/dp_mmfl_image_cache
Train cache: /content/dp_mmfl_image_cache/train
Exists: True


In [ ]:
CACHE_SAMPLE_SIZE = 1000

cache_manifest = (
    train_dataset.manifest
    .sort_values("sample_id")
    .head(CACHE_SAMPLE_SIZE)
    .copy()
)

print("Images selected:", len(cache_manifest))
print(
    "Unique patients:",
    cache_manifest["deid_patient_id"].nunique()
)

print("\nFirst 5:")
display(
    cache_manifest[
        ["sample_id", "deid_patient_id", "path_to_image"]
    ].head()
)

Images selected: 1000
Unique patients: 337

First 5:


,sample_id,deid_patient_id,path_to_image
0,0,patient00003,train/patient00003/study1/view1_frontal.jpg
1,4,patient00009,train/patient00009/study1/view1_frontal.jpg
2,6,patient00016,train/patient00016/study1/view1_frontal.jpg
3,7,patient00021,train/patient00021/study1/view1_frontal.jpg
4,8,patient00026,train/patient00026/study1/view1_frontal.jpg


In [ ]:
def cache_path_for_sample(sample_id):
    return TRAIN_CACHE / f"{int(sample_id):07d}.png"

In [ ]:
from io import BytesIO
from PIL import Image
from tqdm.auto import tqdm

downloaded = 0
failed = []

for _, row in tqdm(
    cache_manifest.iterrows(),
    total=len(cache_manifest),
    desc="Caching images",
):

    output_path = cache_path_for_sample(
        row["sample_id"]
    )

    # Allow safe resume if the notebook disconnects.
    if output_path.exists():
        continue

    try:
        png_path = str(row["path_to_image"])

        if png_path.startswith("train/"):
            png_path = png_path[len("train/"):]

        if png_path.endswith(".jpg"):
            png_path = png_path[:-4] + ".png"

        file = table.file(png_path)
        data = file.read(as_text=False)

        # Validate the image before saving.
        image = Image.open(BytesIO(data))
        image.load()

        if image.format != "PNG":
            raise ValueError(
                f"Unexpected format: {image.format}"
            )

        image.save(
            output_path,
            format="PNG",
        )

        downloaded += 1

    except Exception as e:
        failed.append({
            "sample_id": row["sample_id"],
            "path": row["path_to_image"],
            "error": str(e),
        })

print("\nDownload complete.")
print("Newly downloaded:", downloaded)
print("Failures:", len(failed))

Caching images:   0%|          | 0/1000 [00:00<?, ?it/s]


Download complete.
Newly downloaded: 1000
Failures: 0


In [ ]:
import os

cached_files = list(
    TRAIN_CACHE.glob("*.png")
)

print("Cached PNG files:", len(cached_files))

Cached PNG files: 1000


In [ ]:
assert len(cached_files) == CACHE_SAMPLE_SIZE

print("Cache file-count validation: PASS")

Cache file-count validation: PASS


In [ ]:
total_bytes = sum(
    file.stat().st_size
    for file in cached_files
)

total_mb = total_bytes / (1024 ** 2)
avg_mb = total_mb / len(cached_files)

print(f"Total cache size: {total_mb:.2f} MB")
print(f"Average image size: {avg_mb:.3f} MB")

Total cache size: 2987.34 MB
Average image size: 2.987 MB


In [ ]:
failed

[{'sample_id': 1328,
  'path': 'train/patient01436/study1/view1_frontal.jpg',
  'error': "HTTPSConnectionPool(host='redivis.com', port=443): Read timed out. (read timeout=125)"}]

In [ ]:
from io import BytesIO
from PIL import Image

row = cache_manifest[
    cache_manifest["sample_id"] == 1328
].iloc[0]

png_path = str(row["path_to_image"])

if png_path.startswith("train/"):
    png_path = png_path[len("train/"):]

if png_path.endswith(".jpg"):
    png_path = png_path[:-4] + ".png"

print("Retrying:")
print(png_path)

file = table.file(png_path)
data = file.read(as_text=False)

image = Image.open(BytesIO(data))
image.load()

output_path = cache_path_for_sample(
    row["sample_id"]
)

image.save(
    output_path,
    format="PNG",
)

print("Saved:", output_path)
print("Bytes:", output_path.stat().st_size)
print("Format:", image.format)
print("Size:", image.size)

Retrying:
patient01436/study1/view1_frontal.png
Saved: /content/dp_mmfl_image_cache/train/0001328.png
Bytes: 3800724
Format: PNG
Size: (2828, 2320)


In [ ]:
cached_files = list(TRAIN_CACHE.glob("*.png"))

print("Cached files:", len(cached_files))

assert len(cached_files) == 1000

print("1,000-image cache validation: PASS")

Cached files: 1000
1,000-image cache validation: PASS


Benchmarking cached data

In [ ]:
%%writefile /content/DP-MMFL/src/dp_mmfl/data/cached_image_dataset.py
from pathlib import Path
from typing import Sequence, Union
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import Dataset
from dp_mmfl.data.labels import TARGET_COLUMNS


class CachedCheXpertPlusImageDataset(Dataset):
    """
    Cached image-only PyTorch Dataset for CheXpert Plus frontal radiographs.
    Loads locally stored PNG files indexed by 7-digit zero-padded sample_id.
    """

    def __init__(
        self,
        manifest_path: Union[str, Path],
        cache_root: Union[str, Path],
        sample_ids: Sequence[int],
        image_transform=None,
    ):
        manifest = pd.read_parquet(manifest_path)

        manifest = manifest[
            manifest["sample_id"].isin(sample_ids)
        ].copy()

        # Preserve the exact requested sample order
        order = {
            int(sample_id): i
            for i, sample_id in enumerate(sample_ids)
        }

        manifest["_order"] = manifest["sample_id"].map(order)
        manifest = (
            manifest
            .sort_values("_order")
            .drop(columns="_order")
            .reset_index(drop=True)
        )

        self.manifest = manifest
        self.cache_root = Path(cache_root)
        self.image_transform = image_transform

    def __len__(self) -> int:
        return len(self.manifest)

    def __getitem__(self, index: int):
        row = self.manifest.iloc[index]

        image_path = (
            self.cache_root
            / f"{int(row['sample_id']):07d}.png"
        )

        if not image_path.exists():
            raise FileNotFoundError(
                f"Cached image not found: {image_path}"
            )

        image = Image.open(image_path).convert("L")

        if self.image_transform is not None:
            image = self.image_transform(image)

        labels = torch.tensor(
            [
                0.0
                if pd.isna(row[f"target_{label}"])
                else float(row[f"target_{label}"])
                for label in TARGET_COLUMNS
            ],
            dtype=torch.float32,
        )

        label_mask = torch.tensor(
            [
                float(row[f"mask_{label}"])
                for label in TARGET_COLUMNS
            ],
            dtype=torch.float32,
        )

        return {
            "image": image,
            "labels": labels,
            "label_mask": label_mask,
            "sample_id": row["sample_id"],
            "patient_id": row["deid_patient_id"],
        }

Writing /content/DP-MMFL/src/dp_mmfl/data/cached_image_dataset.py


In [ ]:
%cd /content/DP-MMFL
!git add src/dp_mmfl/data/cached_image_dataset.py
!git commit -m "feat(data): add CachedCheXpertPlusImageDataset for local I/O benchmarking"
!git push origin main

/content/DP-MMFL
[main 6671b47] feat(data): add CachedCheXpertPlusImageDataset for local I/O benchmarking
 1 file changed, 92 insertions(+)
 create mode 100644 src/dp_mmfl/data/cached_image_dataset.py
Enumerating objects: 10, done.
Counting objects: 100% (10/10), done.
Delta compression using up to 2 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (6/6), 1.36 KiB | 695.00 KiB/s, done.
Total 6 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To https://github.com/PreethamHD/DP-MMFL.git
   15fc387..6671b47  main -> main


In [ ]:
from dp_mmfl.data.cached_image_dataset import CachedCheXpertPlusImageDataset

cached_sample_ids = (
    cache_manifest["sample_id"]
    .astype(int)
    .tolist()
)

cached_dataset = CachedCheXpertPlusImageDataset(
    manifest_path=MANIFEST_PATH,
    cache_root=TRAIN_CACHE,
    sample_ids=cached_sample_ids,
    image_transform=get_eval_transforms(),
)

print("Cached dataset size:", len(cached_dataset))
assert len(cached_dataset) == 1000, f"Expected 1,000 samples, found {len(cached_dataset)}"

Cached dataset size: 1000


In [ ]:
cached_sample = cached_dataset[0]

print("Image:    ", cached_sample["image"].shape)
print("Labels:   ", cached_sample["labels"].shape)
print("Mask:     ", cached_sample["label_mask"].shape)
print("Sample ID:", cached_sample["sample_id"])

assert cached_sample["image"].shape == torch.Size([3, 224, 224]), "Image shape mismatch!"
assert cached_sample["labels"].shape == torch.Size([13]), "Labels shape mismatch!"
assert cached_sample["label_mask"].shape == torch.Size([13]), "Label mask shape mismatch!"
assert not torch.isnan(cached_sample["labels"]).any(), "NaN found in cached labels!"
assert not torch.isnan(cached_sample["label_mask"]).any(), "NaN found in cached mask!"
assert torch.all((cached_sample["labels"] == 0.0) | (cached_sample["labels"] == 1.0)), "Non-binary labels!"
assert torch.all((cached_sample["label_mask"] == 0.0) | (cached_sample["label_mask"] == 1.0)), "Non-binary mask!"

print("Cached sample validation: PASS")

Image:     torch.Size([3, 224, 224])
Labels:    torch.Size([13])
Mask:      torch.Size([13])
Sample ID: 0
Cached sample validation: PASS


In [ ]:
import time
import numpy as np
from torch.utils.data import DataLoader

cached_loader = DataLoader(
    cached_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)
print("Cached batches:", len(cached_loader))

NUM_BENCHMARK_BATCHES = min(20, len(cached_loader))
cached_times = []
loader_iter = iter(cached_loader)

for batch_idx in range(NUM_BENCHMARK_BATCHES):
    start = time.perf_counter()

    batch = next(loader_iter)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    elapsed = time.perf_counter() - start
    cached_times.append(elapsed)

    print(f"Batch {batch_idx + 1:02d}/{NUM_BENCHMARK_BATCHES}: {elapsed:.4f}s")

avg_cached_time = np.mean(cached_times)
cached_samples_per_second = 32 / avg_cached_time

print("\n--- Cached Throughput ---")
print(f"Average batch load: {avg_cached_time:.4f}s")
print(f"Samples/sec:        {cached_samples_per_second:.2f}")
print(f"Batches/sec:        {1 / avg_cached_time:.2f}")

Cached batches: 32
Batch 01/20: 2.4043s
Batch 02/20: 2.5144s
Batch 03/20: 2.4123s
Batch 04/20: 2.3185s
Batch 05/20: 2.4081s
Batch 06/20: 2.4458s
Batch 07/20: 2.2819s
Batch 08/20: 2.2866s
Batch 09/20: 2.4671s
Batch 10/20: 2.3070s
Batch 11/20: 2.7677s
Batch 12/20: 2.2977s
Batch 13/20: 2.2956s
Batch 14/20: 2.2781s
Batch 15/20: 2.5054s
Batch 16/20: 2.6381s
Batch 17/20: 2.6827s
Batch 18/20: 2.3169s
Batch 19/20: 2.4423s
Batch 20/20: 2.3601s

--- Cached Throughput ---
Average batch load: 2.4215s
Samples/sec:        13.21
Batches/sec:        0.41


In [ ]:
redivis_time = 23.272
redivis_throughput = 1.38

speedup = redivis_time / avg_cached_time
throughput_improvement = cached_samples_per_second / redivis_throughput

print("=== Data Pipeline Comparison ===")
print(f"Redivis batch time:     {redivis_time:.3f}s")
print(f"Cached batch time:      {avg_cached_time:.4f}s")
print(f"Batch loading speedup:  {speedup:.2f}×")
print(f"Redivis throughput:     {redivis_throughput:.2f} samples/s")
print(f"Cached throughput:      {cached_samples_per_second:.2f} samples/s")
print(f"Throughput improvement: {throughput_improvement:.2f}×")

=== Data Pipeline Comparison ===
Redivis batch time:     23.272s
Cached batch time:      2.4215s
Batch loading speedup:  9.61×
Redivis throughput:     1.38 samples/s
Cached throughput:      13.21 samples/s
Throughput improvement: 9.58×


In [ ]:
for workers in [2, 4]:
    loader = DataLoader(
        cached_dataset,
        batch_size=32,
        shuffle=False,
        num_workers=workers,
        pin_memory=True,
        persistent_workers=True,
    )

    # benchmark 20 batches

In [ ]:
from torch.utils.data import DataLoader
import time
import torch

loader = DataLoader(
    cached_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True,
)

times = []

for i, batch in enumerate(loader):
    if torch.cuda.is_available():
        torch.cuda.synchronize()

    start = time.perf_counter()

    # Transfer to GPU, as we did previously
    images = batch["image"].cuda(non_blocking=True)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    elapsed = time.perf_counter() - start
    times.append(elapsed)

    if i >= 19:
        break

print(f"Average GPU transfer time: {sum(times)/len(times):.4f} sec/batch")

Average GPU transfer time: 0.0026 sec/batch


In [ ]:
loader = DataLoader(
    cached_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True,
)

batch_times = []

start_total = time.perf_counter()

for i, batch in enumerate(loader):
    if torch.cuda.is_available():
        torch.cuda.synchronize()

    now = time.perf_counter()

    if i > 0:
        batch_times.append(now - previous)

    previous = now

    if i >= 20:
        break

total_time = time.perf_counter() - start_total

print(f"Average batch loading time: {sum(batch_times)/len(batch_times):.4f} sec")
print(f"Approx throughput: {32 / (sum(batch_times)/len(batch_times)):.2f} samples/sec")
print(f"Total benchmark time: {total_time:.2f} sec")

Average batch loading time: 1.8060 sec
Approx throughput: 17.72 samples/sec
Total benchmark time: 39.76 sec


In [ ]:
from pathlib import Path
from PIL import Image
import time

source_cache = Path("/content/dp_mmfl_image_cache/train")
preprocessed_cache = Path("/content/dp_mmfl_image_cache_256/train")

preprocessed_cache.mkdir(parents=True, exist_ok=True)

files = sorted(source_cache.glob("*.png"))

print("Source images:", len(files))

start = time.perf_counter()

success = 0
failed = []

for i, src in enumerate(files):
    dst = preprocessed_cache / src.name

    try:
        image = Image.open(src).convert("L")
        image = image.resize((256, 256), Image.Resampling.BILINEAR)
        image.save(dst, format="PNG")

        success += 1

    except Exception as e:
        failed.append((src.name, str(e)))

    if (i + 1) % 100 == 0:
        print(f"Processed {i + 1}/{len(files)}")

elapsed = time.perf_counter() - start

print("\nPreprocessing complete")
print("Success:", success)
print("Failed:", len(failed))
print(f"Time: {elapsed:.2f} sec")

Source images: 1000
Processed 100/1000
Processed 200/1000
Processed 300/1000
Processed 400/1000
Processed 500/1000
Processed 600/1000
Processed 700/1000
Processed 800/1000
Processed 900/1000
Processed 1000/1000

Preprocessing complete
Success: 1000
Failed: 0
Time: 79.39 sec


In [ ]:
files = sorted(preprocessed_cache.glob("*.png"))

print("Cached files:", len(files))

sample = Image.open(files[0])

print("Sample size:", sample.size)
print("Sample mode:", sample.mode)
print("Sample format:", sample.format)

Cached files: 1000
Sample size: (256, 256)
Sample mode: L
Sample format: PNG


CREATING CAHCED DATASET

In [ ]:
from pathlib import Path

import pandas as pd
import torch
from PIL import Image
from torch.utils.data import Dataset

from dp_mmfl.data.labels import TARGET_COLUMNS


class PreprocessedCachedImageDataset(Dataset):
    def __init__(
        self,
        manifest_path,
        cache_root,
        sample_ids,
        image_transform=None,
    ):
        manifest = pd.read_parquet(manifest_path)

        manifest = manifest[
            manifest["sample_id"].isin(sample_ids)
        ].copy()

        order = {
            int(sample_id): i
            for i, sample_id in enumerate(sample_ids)
        }

        manifest["_order"] = manifest["sample_id"].map(order)

        manifest = (
            manifest
            .sort_values("_order")
            .drop(columns="_order")
            .reset_index(drop=True)
        )

        self.manifest = manifest
        self.cache_root = Path(cache_root)
        self.image_transform = image_transform

    def __len__(self):
        return len(self.manifest)

    def __getitem__(self, index):
        row = self.manifest.iloc[index]

        image_path = (
            self.cache_root /
            f"{int(row['sample_id']):07d}.png"
        )

        if not image_path.exists():
            raise FileNotFoundError(
                f"Missing cached image: {image_path}"
            )

        image = Image.open(image_path).convert("L")

        if self.image_transform is not None:
            image = self.image_transform(image)

        labels = torch.tensor(
            [
                0.0 if pd.isna(row[f"target_{label}"])
                else float(row[f"target_{label}"])
                for label in TARGET_COLUMNS
            ],
            dtype=torch.float32,
        )

        label_mask = torch.tensor(
            [
                float(row[f"mask_{label}"])
                for label in TARGET_COLUMNS
            ],
            dtype=torch.float32,
        )

        return {
            "image": image,
            "labels": labels,
            "label_mask": label_mask,
            "sample_id": row["sample_id"],
            "patient_id": row["deid_patient_id"],
        }

In [ ]:
preprocessed_dataset = PreprocessedCachedImageDataset(
    manifest_path=MANIFEST_PATH,
    cache_root="/content/dp_mmfl_image_cache_256/train",
    sample_ids=,
    image_transform=get_train_transforms(),
)

print("Dataset size:", len(preprocessed_dataset))

NameError: name 'train_sample_ids' is not defined

In [ ]:
from torchvision import transforms

# 1. Transforms for images already resized to 256x256 (omit Resize)
cached_train_transform = transforms.Compose([
    transforms.RandomCrop(224),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

# 2. Extract sample IDs directly from cache_manifest
train_sample_ids = cache_manifest["sample_id"].astype(int).tolist()

# 3. Instantiate Dataset
preprocessed_dataset = PreprocessedCachedImageDataset(
    manifest_path=MANIFEST_PATH,
    cache_root="/content/dp_mmfl_image_cache_256/train",
    sample_ids=train_sample_ids,
    image_transform=cached_train_transform,
)

# 4. Validate sample
sample = preprocessed_dataset[0]

print("Dataset size:", len(preprocessed_dataset))
print("Image shape: ", sample["image"].shape)
print("Image dtype: ", sample["image"].dtype)
print("Labels shape:", sample["labels"].shape)
print("Mask shape:  ", sample["label_mask"].shape)

assert len(preprocessed_dataset) == 1000
assert sample["image"].shape == (3, 224, 224)
assert sample["labels"].shape == (13,)
assert sample["label_mask"].shape == (13,)
print("\nValidation check: PASS")

Dataset size: 1000
Image shape:  torch.Size([3, 224, 224])
Image dtype:  torch.float32
Labels shape: torch.Size([13])
Mask shape:   torch.Size([13])

Validation check: PASS


In [ ]:
import time
from torch.utils.data import DataLoader

benchmark_loader = DataLoader(
    preprocessed_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True,
)

iterator = iter(benchmark_loader)
batch_times = []

start_total = time.perf_counter()

for i in range(20):
    start = time.perf_counter()
    batch = next(iterator)
    batch_times.append(time.perf_counter() - start)

total_time = time.perf_counter() - start_total

# Exclude the first batch to reduce worker-startup effects.
steady_times = batch_times[1:]

avg_batch = sum(steady_times) / len(steady_times)

print(f"Batches measured: {len(batch_times)}")
print(f"First batch time: {batch_times[0]:.4f} sec")
print(f"Average batch loading time (excluding first): {avg_batch:.4f} sec")
print(f"Approx throughput: {32 / avg_batch:.2f} samples/sec")
print(f"Total benchmark time: {total_time:.2f} sec")

Batches measured: 20
First batch time: 0.2701 sec
Average batch loading time (excluding first): 0.0823 sec
Approx throughput: 388.87 samples/sec
Total benchmark time: 1.83 sec
